<a href="https://colab.research.google.com/github/Deepthi169/Structured-large-scale-data-generation/blob/main/copro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics open_clip_torch transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import random
import shutil

BASE_PATH = "/content/drive/MyDrive/copro/data"

RAW_PATH = os.path.join(BASE_PATH, "raw")
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH = os.path.join(BASE_PATH, "val")

classes = ["neem", "tulasi"]

for cls in classes:

    raw_class = os.path.join(RAW_PATH, cls)
    train_class = os.path.join(TRAIN_PATH, cls)
    val_class = os.path.join(VAL_PATH, cls)

    os.makedirs(train_class, exist_ok=True)
    os.makedirs(val_class, exist_ok=True)

    images = [
        f for f in os.listdir(raw_class)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    random.shuffle(images)

    split_index = int(0.8 * len(images))

    train_images = images[:split_index]
    val_images = images[split_index:]

    # Copy training images
    for img in train_images:
        shutil.copy(
            os.path.join(raw_class, img),
            os.path.join(train_class, img)
        )

    # Copy validation images
    for img in val_images:
        shutil.copy(
            os.path.join(raw_class, img),
            os.path.join(val_class, img)
        )

    print(f"{cls} → Train: {len(train_images)} | Val: {len(val_images)}")

print("✅ 80:20 dataset split completed successfully")

neem → Train: 116 | Val: 30
tulasi → Train: 116 | Val: 30
✅ 80:20 dataset split completed successfully


In [ ]:
from ultralytics import YOLO
import torch

model = YOLO("yolov8m-cls.yaml")

model.train(
    data="/content/drive/MyDrive/copro/data",
    epochs=25,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLOv8m-cls summary: 80 layers, 17,053,336 parameters, 17,053,336 gradients, 42.9 GFLOPs
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/copro/data, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscrip

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c3eb1bb4ec0>
curves: []
curves_results: []
fitness: 0.9977272748947144
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.9954545497894287, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9977272748947144}
save_dir: PosixPath('/content/runs/classify/train')
speed: {'preprocess': 0.5972349272724387, 'inference': 8.146882218182904, 'loss': 0.0004079636369169748, 'postprocess': 0.0013796000003325885}
task: 'classify'
top1: 0.9954545497894287
top5: 1.0

In [ ]:
import os
import json
import cv2
import numpy as np
from ultralytics import YOLO
import open_clip
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

In [ ]:
class QualityMetrics:

    def compute(self, image_path):

        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        blur = cv2.Laplacian(img, cv2.CV_64F).var()
        brightness = np.mean(img) / 255
        contrast = np.std(img) / 255

        return {
            "blur_score": round(float(blur),3),
            "brightness": round(float(brightness),3),
            "contrast": round(float(contrast),3)
        }

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

class YOLOExtractor:

    def __init__(self):

        # Your trained classification model
        self.model = YOLO("/content/runs/classify/train/weights/best.pt")

        self.conf = 0.05


    def extract(self, image_path):

        objects = []

        image = cv2.imread(image_path)

        if image is None:
            return objects

        h, w = image.shape[:2]
        image_area = h * w

        results = self.model.predict(
            source=image_path,
            conf=self.conf,
            imgsz=640,
            verbose=False
        )

        for r in results:

            if r.probs is None:
                continue

            class_id = int(r.probs.top1)
            confidence = float(r.probs.top1conf)

            label = self.model.names[class_id]

            # Whole image as bounding box
            x1, y1, x2, y2 = 0, 0, w, h

            bbox_area = w * h
            area_ratio = bbox_area / image_area

            # Calculate green ratio
            hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

            lower_green = np.array([35,40,40])
            upper_green = np.array([85,255,255])

            mask = cv2.inRange(hsv, lower_green, upper_green)

            green_pixels = np.sum(mask > 0)
            total_pixels = h * w

            green_ratio = green_pixels / total_pixels

            # Leaf stage estimation
            if green_ratio > 0.6:
                leaf_stage = "Mature"
            elif green_ratio > 0.3:
                leaf_stage = "Young"
            else:
                leaf_stage = "Old"

            objects.append({
                "label": label,
                "confidence": round(confidence,3),
                "bbox": [x1,y1,x2,y2],
                "area_ratio": round(float(area_ratio),3),
                "green_ratio": round(float(green_ratio),3),
                "leaf_stage": leaf_stage
            })

        return objects

In [ ]:
class CLIPExtractor:

    def __init__(self):

        self.model,_,self.preprocess = open_clip.create_model_and_transforms(
            "ViT-B-32", pretrained="openai"
        )

        self.model.eval()

    def extract(self,image_path):

        image = self.preprocess(Image.open(image_path)).unsqueeze(0)

        with torch.no_grad():
            embedding = self.model.encode_image(image)

        return embedding.cpu().numpy().flatten().tolist()

In [ ]:
class BLIPExtractor:

    def __init__(self):

        self.processor = BlipProcessor.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        )

        self.model = BlipForConditionalGeneration.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        )

    def extract(self,image_path):

        image = Image.open(image_path).convert("RGB")

        inputs = self.processor(image, return_tensors="pt")

        out = self.model.generate(**inputs)

        caption = self.processor.decode(out[0], skip_special_tokens=True)

        return caption

In [ ]:
class FeatureFusion:

    def fuse(self,image_name,quality,clip_emb,caption,objects):

        return {
            "image":image_name,

            "quality_metrics":quality,

            "semantic_features":{
                "clip_embedding":clip_emb
            },

            "descriptive_features":{
                "caption":caption
            },

            "object_features":{
                "detected_objects":objects,
                "object_count":len(objects)
            }
        }

In [ ]:
quality_checker = QualityMetrics()

yolo_extractor = YOLOExtractor()

clip_extractor = CLIPExtractor()

blip_extractor = BLIPExtractor()

fusion_engine = FeatureFusion()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

In [ ]:
import os
import json

DATASET_DIR = "/content/drive/MyDrive/copro/data/raw"
OUTPUT_DIR = "/content/drive/MyDrive/copro/output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
final_dataset = []

for root, dirs, files in os.walk(DATASET_DIR):

    for img_name in files:

        if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        image_path = os.path.join(root, img_name)

        print("Processing:", img_name)

        quality = quality_checker.compute(image_path)

        clip_emb = clip_extractor.extract(image_path)

        caption = blip_extractor.extract(image_path)

        objects = yolo_extractor.extract(image_path)

        structured = fusion_engine.fuse(
            img_name,
            quality,
            clip_emb,
            caption,
            objects
        )

        final_dataset.append(structured)

Processing: 1803.jpg
Processing: 1812.jpg
Processing: 1787.jpg
Processing: 1830.jpg
Processing: 1831.jpg
Processing: 1834.jpg
Processing: 1806.jpg
Processing: 1828.jpg
Processing: 1810.jpg
Processing: 1788.jpg
Processing: 1807.jpg
Processing: 1805.jpg
Processing: 1818.jpg
Processing: 1794.jpg
Processing: 1808.jpg
Processing: 1789.jpg
Processing: 1786.jpg
Processing: 1819.jpg
Processing: 1802.jpg
Processing: 1832.jpg
Processing: 1804.jpg
Processing: 1822.jpg
Processing: 1799.jpg
Processing: 1785.jpg
Processing: 1790.jpg
Processing: 1797.jpg
Processing: 1781.jpg
Processing: 1816.jpg
Processing: 1833.jpg
Processing: 1811.jpg
Processing: 1823.jpg
Processing: 1782.jpg
Processing: 1835.jpg
Processing: 1825.jpg
Processing: 1793.jpg
Processing: 1824.jpg
Processing: 1815.jpg
Processing: 1817.jpg
Processing: 1795.jpg
Processing: 1796.jpg
Processing: 1827.jpg
Processing: 1821.jpg
Processing: 1792.jpg
Processing: 1780.jpg
Processing: 1783.jpg
Processing: 1801.jpg
Processing: 1826.jpg
Processing: 1

In [ ]:
OUTPUT_JSON = os.path.join(
    OUTPUT_DIR,
    "plant_multimodal_dataset.json"
)

with open(OUTPUT_JSON, "w") as f:
    json.dump(final_dataset, f, indent=4)

print("JSON saved at:", OUTPUT_JSON)
print("Total images processed:", len(final_dataset))

JSON saved at: /content/drive/MyDrive/copro/output/plant_multimodal_dataset.json
Total images processed: 292
